In [2]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH100=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH100_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH100_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH100[mask]

from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 50 < x < 150]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=100, sigma=5, gamma=1, norm2=1, mu2=100, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (50, 150)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()

/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 105.4 (χ²/ndof = 2.5)      │              Nfcn = 491              │
│ EDM = 8.97e-07 (Goal: 0.0002)    │            time = 0.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.767   │   0.019   │            │            │         │         │       │
│ 1 │ mu     │  103.41   │   0.09    │            │            │   50    │   150   │       │
│ 2 │ sigma  │   5.71    │   0.15    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │   3.17    │   0.14    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.263   │   0.018   │            │            │         │         │       │
│ 5 │ mu2    │   95.5    │    0.6    │            │            │         │         │       │
│ 6 │ sigma2 │   13.80   │   0.31    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────────────┐
│        │     norm       mu    sigma    gamma    norm2      mu2   sigma2   gamma2 │
├────────┼─────────────────────────────────────────────────────────────────────────┤
│   norm │  0.00037  -0.2e-3   1.4e-3   0.6e-3 -0.33e-3  -9.3e-3  -1.4e-3        0 │
│     mu │  -0.2e-3  0.00743   -0.005    0.004  0.26e-3   -0.005   -0.016    0.000 │
│  sigma │   1.4e-3   -0.005   0.0228   -0.014 -1.43e-3   -0.037    0.020    0.000 │
│  gamma │   0.6e-3    0.004   -0.014   0.0195 -0.39e-3   -0.012   -0.033     0.00 │
│  norm2 │ -0.33e-3  0.26e-3 -1.43e-3 -0.39e-3 0.000321  8.82e-3  1.02e-3        0 │
│    mu2 │  -9.3e-3   -0.005   -0.037   -0.012  8.82e-3    0.305     0.06      0.0 │
│ sigma2 │  -1.4e-3   -0.016    0.020   -0.033  1.02e-3     0.06   0.0979      0.0 │
│ gamma2 │        0    0.000    0.000     0.00        0      0.0      0.0        0 │
└────────┴─────────────────────────────────────────────────────────────────────────┘

In [3]:
fit_MH100_values={}
fit_MH100_errors={}

fit_values={'MH100': fit_MH100_values,}
fit_errors={'MH100_errors': fit_MH100_errors}



for param in m_voigt.parameters:
    fit_MH100_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH100_errors[error] = m_voigt.errors[error]

print(fit_MH100_values)
print(fit_MH100_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH100"]=fit_MH100_values
results["MH100_errors"]=fit_MH100_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH100"]=fit_MH100_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH100_errors"]=fit_MH100_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.7667105898426677, 'mu': 103.41498730415057, 'sigma': 5.707662064168594, 'gamma': 3.1695488679939436, 'norm2': 0.26287455944987237, 'mu2': 95.46191785407376, 'sigma2': 13.796789530506208, 'gamma2': 0.001}
{'norm': 0.019246456510648685, 'mu': 0.08617069306048109, 'sigma': 0.151025955672202, 'gamma': 0.13973922523926996, 'norm2': 0.01792439027204349, 'mu2': 0.5521819173258976, 'sigma2': 0.3129291580889273, 'gamma2': 1e-05}
